# 01.02 - Loss Functions Deconstructed

**Module:** 01 - ML Fundamentals  
**Subtitle:** From signed residuals to point-wise losses to aggregate summaries  
**Prerequisites:** 01.01 - The Learning Paradigm  
**Difficulty:** introductory / foundational  
**Estimated time:** 45-60 minutes

This lesson uses synthetic, deterministic, dimensionless values. It opens the two summary numbers from 01.01 into per-example losses and their means.

**Outcomes:** classify residuals and losses; compute absolute and squared losses; derive MAE and MSE; explain sign loss and the crossover at `|e|=1`; distinguish evaluation from training; and reject unsupported real-world, causal, robustness, or generalisation conclusions.


## 1. The Problem

A residual is signed, but a summary such as MAE or MSE is non-negative. What happens between the residual and one summary number, and why can absolute and squared summaries respond differently to the same residuals?

In 01.01 you computed residuals and summary numbers. This lesson opens those numbers up: each is the mean of per-example losses. Computing a loss evaluates a fixed set; it does not train a model.


## 2. Toy Example or Dataset

All values below are synthetic, deterministic, dimensionless, and use `N=4`. The three residual vectors are owned by this lesson:

| vector | residuals |
|---|---|
| A (baseline) | `[0.5, 1.0, 1.5, 2.0]` |
| B (large-residual) | `[0.5, 1.0, 1.5, 5.0]` |
| C (signed) | `[-0.5, 1.5, -2.0, 1.0]` |

Notation: `e_i` is a signed residual, `l_abs(e_i)=|e_i|` is an absolute point-wise loss, `l_sq(e_i)=e_i^2` is a squared point-wise loss, and `L=(1/N) sum(l_i)` is an aggregate loss. This lesson uses the plain `1/N` mean.

`residual != loss`; `point-wise loss != aggregate loss`; `evaluation metric != training procedure`.


In [ ]:
import numpy as np

RTOL = 0.0
ATOL = 1e-12
A = np.array([0.5, 1.0, 1.5, 2.0], dtype=np.float64)
B = np.array([0.5, 1.0, 1.5, 5.0], dtype=np.float64)
C = np.array([-0.5, 1.5, -2.0, 1.0], dtype=np.float64)


## 3. Intuition Before Math

Absolute loss keeps magnitude linearly: `|e|`. Squared loss removes sign and squares magnitude: `e^2`. Both are non-negative and equal zero only when `e=0`. Neither preserves whether the prediction was above or below the target.

Vector C has mean residual `(-0.5 + 1.5 - 2.0 + 1.0)/4 = 0`, but its errors are not zero: `MAE_C=1.25` and `MSE_C=1.875`. Vectors A and C have the same aggregate values despite different signed residuals.


## 4. Mathematical Derivation

For residual `e_i = y_i - y_hat_i`, define point-wise losses `l_abs(e_i)=|e_i|` and `l_sq(e_i)=e_i^2`. Their aggregate means are

`MAE = (1/N) sum |e_i|` and `MSE = (1/N) sum e_i^2`.

The crossover follows without calculus: `e^2 - |e| = |e|(|e|-1)`. Thus `e^2 < |e|` for `0<|e|<1`, equality holds at `|e|=1`, and `e^2>|e|` for `|e|>1`. For target units `u`, residual and MAE have units `u`; squared loss and MSE have units `u^2`.

**Frozen stance:** MAE and MSE are two different summaries of the same residuals. MAE changes proportionally to changes in `|e|`; MSE changes proportionally to changes in `e^2`. Neither is universally preferable: the appropriate summary depends on the task and on the objective chosen by the practitioner. This lesson computes both and compares their behaviour on frozen teaching values only.


## 5. Manual Walkthrough

Every point-wise value and addition is explicit.

**Vector A**

- Absolute: `|0.5|=0.5`, `|1.0|=1.0`, `|1.5|=1.5`, `|2.0|=2.0`.
- Squared: `0.5^2=0.25`, `1.0^2=1.00`, `1.5^2=2.25`, `2.0^2=4.00`.
- Sums: `0.5+1.0+1.5+2.0=5.0`; `0.25+1.00+2.25+4.00=7.50`.
- Means: `MAE=5.0/4=1.25`; `MSE=7.5/4=1.875`.

**Vector B**

- Absolute: `0.5`, `1.0`, `1.5`, `5.0`; sum `0.5+1.0+1.5+5.0=8.0`; `MAE=8.0/4=2.0`.
- Squared: `0.25`, `1.00`, `2.25`, `25.00`; sum `0.25+1.00+2.25+25.00=28.50`; `MSE=28.5/4=7.125`.

**Vector C**

- Absolute: `0.5`, `1.5`, `2.0`, `1.0`; sum `0.5+1.5+2.0+1.0=5.0`; `MAE=5.0/4=1.25`.
- Squared: `0.25`, `2.25`, `4.00`, `1.00`; sum `0.25+2.25+4.00+1.00=7.50`; `MSE=7.5/4=1.875`.

Changing A's fourth residual from `2.0` to `5.0` changes absolute loss by `3.0`, so `Delta MAE=3.0/4=0.75`; squared loss changes by `21.0`, so `Delta MSE=21.0/4=5.25`. The ratios on these vectors are x2.5 for the residual, x6.25 for its squared loss, x1.6 for MAE, and x3.8 for MSE.


## 6. Teaching Implementation

The functions validate non-empty finite one-dimensional float64 arrays, compute point-wise arrays first, and then take their means. No parameters are fitted and no optimizer is run.


In [ ]:
def validate_residuals(residuals):
    if not isinstance(residuals, np.ndarray):
        raise TypeError("residuals must be a numpy.ndarray")
    if residuals.dtype != np.float64:
        raise ValueError("residuals must have dtype float64")
    if residuals.ndim != 1:
        raise ValueError("residuals must be a 1-D array")
    if residuals.size == 0:
        raise ValueError("residuals must be non-empty")
    if not np.all(np.isfinite(residuals)):
        raise ValueError("residuals must contain only finite values")

def pointwise_absolute_loss(residuals):
    validate_residuals(residuals)
    absolute_losses = np.abs(residuals)
    return absolute_losses

def pointwise_squared_loss(residuals):
    validate_residuals(residuals)
    squared_losses = residuals ** 2
    return squared_losses

def mean_absolute_error(residuals):
    absolute_losses = pointwise_absolute_loss(residuals)
    return np.mean(absolute_losses)

def mean_squared_error(residuals):
    squared_losses = pointwise_squared_loss(residuals)
    return np.mean(squared_losses)

summary = {
    "A": (mean_absolute_error(A), mean_squared_error(A)),
    "B": (mean_absolute_error(B), mean_squared_error(B)),
    "C": (mean_absolute_error(C), mean_squared_error(C)),
}
summary


## 7. Verification

The frozen shapes, float64 dtype, point-wise values, non-negativity, scalar outputs, finite outputs, corrected aggregates, A/C equality, C's zero mean, A-to-B changes, and validation errors are checked with `rtol=0.0` and `atol=1e-12`.


In [ ]:
expected = {"A": (1.25, 1.875), "B": (2.0, 7.125), "C": (1.25, 1.875)}
for name, residuals in {"A": A, "B": B, "C": C}.items():
    assert residuals.shape == (4,) and residuals.dtype == np.float64
    assert np.all(pointwise_absolute_loss(residuals) >= 0)
    assert np.all(pointwise_squared_loss(residuals) >= 0)
    np.testing.assert_allclose(summary[name], expected[name], rtol=RTOL, atol=ATOL)
    assert isinstance(mean_absolute_error(residuals), np.float64)
    assert mean_absolute_error(residuals).ndim == 0
np.testing.assert_allclose(np.mean(C), 0.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose((5.0 - 2.0) / 4, 0.75, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose((25.0 - 4.0) / 4, 5.25, rtol=RTOL, atol=ATOL)
print("verification: PASS")


## 8. Independent Cross-Check

The following scalar loop uses separate Python lists and builtin arithmetic. It does not call the primary functions. The standalone verifier also includes literal expressions for all three vectors and validation edge cases.


In [ ]:
lists = {"A": [0.5, 1.0, 1.5, 2.0], "B": [0.5, 1.0, 1.5, 5.0], "C": [-0.5, 1.5, -2.0, 1.0]}
for name, values in lists.items():
    abs_total = 0.0
    sq_total = 0.0
    for value in values:
        abs_total += abs(value)
        sq_total += value * value
    np.testing.assert_allclose([abs_total / len(values), sq_total / len(values)], expected[name], rtol=RTOL, atol=ATOL)
print("independent arithmetic: PASS")


## 9. Visual Explanation

![Static loss visual](assets/loss_functions_visual.png)

**Caption:** FIG-01-02-01 compares absolute and squared point-wise losses for synthetic vectors A and B, then plots both loss functions against signed residuals. The figure uses labels, hatching, line styles, and markers so colour is not the only encoding. All values are dimensionless and no model or performance claim is made.

**Alt text:** Three panels show absolute and squared losses for A and B and the two symmetric loss curves. A uses absolute values 0.5, 1.0, 1.5, 2.0 and squared values 0.25, 1.00, 2.25, 4.00. B replaces 2.0 with 5.0, giving squared value 25.00. The curves cross at residual magnitude 1; at residual 5 they reach 5 and 25.


## 10. Failure Modes

- A residual is not a loss, and a point-wise loss is not an aggregate.
- Negative residuals are valid; absolute and squared losses discard their sign.
- Mean residual zero does not mean zero error: C is the counterexample.
- MSE is not MAE squared: `1.25^2=1.5625`, not `1.875`.
- Squaring is not always larger: `0.5^2=0.25`.
- Computing a summary on fixed values is evaluation, not training.
- These values do not establish robustness, real-world quality, causality, or generalisation.


## 11. Exercises

1. Classify `e_3=-2.0`, `|e_3|=2.0`, `e_3^2=4.0`, `MAE_C=1.25`, and `MSE_C=1.875`.
2. Reproduce every sum and mean for Vector B.
3. Derive the A-to-B changes in MAE and MSE.
4. Identify the three crossover regimes using 0.5, 1.0, and 1.5.
5. Explain how C has mean residual zero but positive MAE.
6. State the units of residual, MAE, squared loss, and MSE.
7. Explain why computing MSE is not training.


## 12. References

- Hastie, Tibshirani, and Friedman, *The Elements of Statistical Learning*, §2.4: https://hastie.su.domains/ElemStatLearn/
- James et al., *An Introduction to Statistical Learning*, DOI 10.1007/978-3-031-38747-0: https://www.statlearning.com/
- Goodfellow, Bengio, and Courville, *Deep Learning*, §§5.1.2 and 5.1.4: https://www.deeplearningbook.org/
- Deisenroth, Faisal, and Ong, *Mathematics for Machine Learning*, Ch. 9: https://mml-book.github.io/
- Murphy, *Probabilistic Machine Learning: An Introduction*, §4.3: https://probml.github.io/pml-book/book1.html
- NumPy 1.26 documentation: `absolute`, `mean`, and `testing.assert_allclose`.
- Bishop, *Pattern Recognition and Machine Learning*, §§1.1 and 1.5.5 (source recheck required before final publication).
